In [6]:
import pandas as pd

In [7]:
df = pd.read_csv('birthdays.csv')
df.head()

,Timestamp,Full Name:,WhatsApp Number:,Date of Birth,Email Address,Sex,Name on WhatsApp
0,10/9/2026 18:13:53,Olanipekun Olumide Sunday,08088608496,9/19,olanipekunolumidesunday@gmail.com,Male,Gauss
1,10/9/2026 18:31:37,Paul,09041501470,5/12,adewolepaul513@gmail.com,Male,FBBL
2,10/10/2026 9:01:15,Akinola Oluwaferanmi Taiwo,+2347039310019,6/9,taiwoakinola059@gmail.com,Female,Ella Honey
3,10/10/2026 9:30:24,Suyanbola folasade best,07033557664,5/31,gbemisolabest@gmail.com,Female,Folasade
4,10/11/2026 18:46:39,Obatunde Faidat Oluwamayowa,8147260902,5/8,obatundefaidatforoau@gmail.com,Female,Adeola


### What this data is for?

This data is meant to feed an agent with information about the birthday date of our department student. Details about their month, date of birth are provided and their whatsapp number including their nickname to be celebrated as.

### What question would the agent be asked and how is it meant to answer? 

user: who is having birthday today or who are those having birthday this month?

agent: list the people having birthday that month or that day.

user: provide me the details of those having birthday today or this month 

agent: femi is having birthday on nov 2, whatsapp no:+234555555, nicknmame:femzy

#### 1. Knowing the columns

| Data | DIctionary |
| -- | -- |
| Timestamp | when each rows were recorded |
| Full Name | Name of the respondent|
| Whatsapp Number | The whatsapp contact of the respondent|
| Date of birth | The birthdate of the respondent | 
| Email address | Email address of the respondent |
| sex | The gender of the respondent | 
| Name on whatsapp | The nickname of the respondent |

In [8]:
audit = pd.DataFrame(index=df.columns)
audit['Data_types'] = df.dtypes.astype(str)
audit['missing rows'] = df.isna().sum()
audit['available rows'] = df.notna().sum()
audit['distinct'] = df.nunique().sum()
display(audit)

,Data_types,missing rows,available rows,distinct
Timestamp,str,0,74,434
Full Name:,str,0,74,434
WhatsApp Number:,str,0,74,434
Date of Birth,str,0,74,434
Email Address,str,0,74,434
Sex,str,0,74,434
Name on WhatsApp,str,0,74,434


### 2. Correct column names 

In [9]:
# rename all columns
df.rename(columns={
    'Full Name:':'Name',
    'WhatsApp Number:': 'whatsapp_no',
    'Date of Birth': 'date_of_birth',
    'Email Address': 'email_address',
    'Sex':'gender',
    'Name on WhatsApp':'nickname',
}, inplace=True)

df.columns


/tmp/ipykernel_1689/1099337144.py:2: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={


Index(['Timestamp', 'Name', 'whatsapp_no', 'date_of_birth', 'email_address',
       'gender', 'nickname'],
      dtype='str')

### What columns are useful to our agent?

Name, whatsapp no, date of birth, and nickname

### 3. Inspect the needed columns

some respondent recorded only their first name while some recorded their first/other/last name in no order format

In [10]:
### check how name of respondent are recorded 
df['Name'].value_counts()
### did anyone recorded their nickname eactly as ther name
check = pd.DataFrame()
check['name'] = df['Name']
check['nickname'] = df['nickname']
display(check)

,name,nickname
0,Olanipekun Olumide Sunday,Gauss
1,Paul,FBBL
2,Akinola Oluwaferanmi Taiwo,Ella Honey
3,Suyanbola folasade best,Folasade
4,Obatunde Faidat Oluwamayowa,Adeola
...,...,...
69,Orifunmilade Julius Toluwani,Jay might
70,Odewumi Glory Mary,Marynikky
71,Adebayo Hamzat,DEE WÊAŁTH💰😘
72,Taiwo Kehinde Elizabeth,Taiwo Oluwabusayo


### 4. Check date of birth

In [11]:
# 1. Convert the 'month/date' string into a real pandas datetime object
# %m represents the month, %d represents the day.
# errors='coerce' ensures that if someone typed "idk", it turns into NaT (Not a Time) instead of crashing your code.
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], format='%m/%d', errors='coerce')

# 2. Extract and format it to 'date-month' (DD-MM) as a clean string
df['date_of_birth'] = df['date_of_birth'].dt.strftime('%d-%m')

# Verify the transformation
df['date_of_birth'].head()

0    19-09
1    12-05
2    09-06
3    31-05
4    08-05
Name: date_of_birth, dtype: str

### 5. Check if there are any double submission 

In [12]:
# keep=False tells pandas to show EVERY instance of the duplicate, 
# not just the first or last one.
duplicates = df[df.duplicated(subset=['email_address'], keep=False)]

# Sort by email so the duplicated rows sit right next to each other
display(duplicates.sort_values('email_address'))

# Ensure the data is sorted chronologically so the newest submission is at the bottom
df = df.sort_values('Timestamp')

# Drop the older duplicate based on their Email (or Full Name), keeping the latter
df = df.drop_duplicates(subset=['email_address'], keep='last')

# Reset the index so your row numbers stay perfectly sequential after dropping the row
df = df.reset_index(drop=True)

print(f"Total unique students after deduplication: {len(df)}")

,Timestamp,Name,whatsapp_no,date_of_birth,email_address,gender,nickname
8,10/12/2026 11:52:55,Ajayi Jesuloba Dorcas,09048150608,05-01,jesulobaajayi853@gmail.com,Female,Grãçé🔥🔥
62,10/20/2026 8:29:39,Ajayi Jesuloba Dorcas,09048150608,05-01,jesulobaajayi853@gmail.com,Female,Gràçë🔥🔥🔥


Total unique students after deduplication: 73


In [13]:
def clean_phone(phone):
    phone = str(phone).strip().replace(" ", "").replace("-", "")
    if phone in ['nan', 'None', '']: return None
    
    # Standardize Nigerian numbers to start with +234
    if phone.startswith('+234'): return phone
    if phone.startswith('0'): return f"+234{phone[1:]}"
    if phone.startswith('234'): return f"+{phone}"
    return f"+234{phone}"

df['whatsapp_no'] = df['whatsapp_no'].apply(clean_phone)

# Check how many invalid numbers slipped through (e.g., numbers that are too short)
invalid_numbers = df[df['whatsapp_no'].str.len() < 13]
print("Flagged numbers for manual review:")
display(invalid_numbers[['Name', 'whatsapp_no']])

Flagged numbers for manual review:


,Name,whatsapp_no


In [14]:
# Check the total missing values per column
missing_data = df.isna().sum()
print("Missing Data Points:")
print(missing_data[missing_data > 0])

# Drop anyone who doesn't have a valid Date of Birth (the agent can't work without this)
df = df.dropna(subset=['date_of_birth'])

Missing Data Points:
Series([], dtype: int64)


In [15]:
# Keep only the essential columns for the PRO
final_cols = ['Name', 'nickname', 'whatsapp_no', 'date_of_birth']
df_clean = df[final_cols].copy()

# Save the pristine data for the agent to use
df_clean.to_csv('birthdays_clean.csv', index=False)

print(f"Cleaned {len(df_clean)} records. Ready for the PRO agent!")

Cleaned 73 records. Ready for the PRO agent!
